# Study notebook: `generation_safety.py`

This notebook contains the exact production code from the matching Python module. Each explanation appears immediately before the code section it explains. The code is copied line-for-line from the application module.

## `imports and module setup`

This section is part of the production implementation and supports the module's main workflow.

In [ ]:
"""Safe generation wrapper for Groq or another OpenAI-compatible provider."""


## `imports and module setup`

This section is part of the production implementation and supports the module's main workflow.

In [ ]:
from __future__ import annotations



## `imports and module setup`

This section is part of the production implementation and supports the module's main workflow.

In [ ]:
import re


## `imports and module setup`

This section is part of the production implementation and supports the module's main workflow.

In [ ]:
from typing import Callable



## `imports and module setup`

This section is part of the production implementation and supports the module's main workflow.

In [ ]:
try:
    from .augmentation import prompt_messages
except ImportError:  # notebook / direct script execution from rag/
    from augmentation import prompt_messages




## `imports and module setup`

This section is part of the production implementation and supports the module's main workflow.

In [ ]:
UNSAFE = ("how to forge", "how to loot", "how to steal", "make a weapon", "harm a person")




## `safety_check`

Blocks unsafe requests before they reach an external model and validates the input length.

In [ ]:
def safety_check(question: str) -> tuple[bool, str]:
    # Block harmful requests before they reach the external AI provider.
    normalized = question.casefold().strip()
    for phrase in UNSAFE:
        if phrase in normalized:
            return False, "I can help with the historical or archaeological context, but not instructions for harm, theft, or forgery."
    if len(question) > 2000:
        return False, "Please shorten the question to 2,000 characters or fewer."
    return True, ""




## `generate`

Runs the safety gate, prepares grounded messages, or calls the supplied provider function.

In [ ]:
async def generate(question: str, records: list[dict], chat_call: Callable | None = None) -> dict:
    # Safety comes first; generation starts only after the question is allowed.
    allowed, message = safety_check(question)
    if not allowed:
        return {"response": message, "citations": [], "blocked": True}
    if chat_call is None:
        return {"messages": prompt_messages(question, records), "blocked": False}
    result = await chat_call(prompt_messages(question, records))
    # Keep accidental unsupported citation claims visible to the caller.
    citations = sorted(set(re.findall(r"\[S\d+\]", result)))
    return {"response": result, "citations": citations, "blocked": False}

